# Day 13 — App Cohort Retention, Churn Risk & Lifetime Value (LTV) Forecasting Engine

**Project**: Google Play Store Ecosystem Analysis & NLP Autocomplete/Autocorrect Analytics  
**Milestone**: Day 13 (Parametric Retention Decay, LTV Forecasting, Abandonware Risk & Markowitz Portfolio Allocator)  
**Focus**: Weibull & Power-Law survival curves, discounted cash flow LTV modeling, developer maintenance recency, 0–100 churn risk index, and risk-adjusted capital allocation.

---

### Mathematical & Methodological Highlights:
1. **Parametric Retention Curves**: Power-law decay $R(t) = R_0 \cdot t^{-\alpha}$ and Weibull survival $S(t) = \exp(-(t/\lambda)^\beta)$ calibrated across 33 categories.
2. **Customer Lifetime Value (LTV)**: Discounted cumulative cash flow $LTV = \text{NetPrice} + \sum_{t=1}^T \frac{R(t) \cdot \text{ARPDAU}}{(1 + d/365)^t}$ with CAC payback analysis.
3. **Developer Maintenance Lifecycle**: Recency analysis relative to Play Store baseline (2018-08-08), segmenting into Vibrant, Mature, Stagnant, and Zombie Abandonware.
4. **Composite Churn Risk Index (0–100)**: Multi-pillar risk synthesis combining update stagnancy (30%), negative review polarity (25%), rating degradation (20%), volatility (15%), and APK bloat (10%).
5. **Modern Portfolio Theory (Markowitz)**: Quadratic mean-variance optimization allocating capital across app verticals to maximize risk-adjusted Sharpe ratios.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is accessible
WORKSPACE_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

from src.retention_churn_ltv_engine import RetentionChurnLTVEngine

engine = RetentionChurnLTVEngine()
print("Catalog apps loaded:", len(engine.df_apps))
print("Cleaned reviews loaded:", len(engine.df_reviews))


## 1. Parametric User Retention Decay & Milestone Benchmarks

Evaluate Power-Law vs. Weibull survival trajectories and milestone retention ($D_1, D_7, D_{14}, D_{30}, D_{90}, D_{360}$) across mobile app categories.


In [ ]:
target_cats = ["GAME", "SOCIAL", "FINANCE", "TOOLS", "HEALTH_AND_FITNESS"]
ret_records = []
for cat in target_cats:
    res = engine.estimate_retention_curve(cat, is_category=True, days=360)
    ret_records.append({
        "Category": cat,
        "D1 (%)": round(res.milestones.day_1 * 100, 1),
        "D7 (%)": round(res.milestones.day_7 * 100, 1),
        "D14 (%)": round(res.milestones.day_14 * 100, 1),
        "D30 (%)": round(res.milestones.day_30 * 100, 1),
        "D90 (%)": round(res.milestones.day_90 * 100, 1),
        "D360 (%)": round(res.milestones.day_360 * 100, 1),
        "Decay Alpha": res.decay_parameter,
        "Cumulative Active Days (AUC)": res.retention_auc
    })
pd.DataFrame(ret_records)


## 2. Customer Lifetime Value (LTV) Forecasting & Payback Unit Economics

Simulate 360-day discounted cash flows, ARPDAU capitalization, CAC breakeven payback periods, and LTV/CAC ratios.


In [ ]:
sample_apps = ["Subway Surfers", "Duolingo: Learn Languages Free", "Clash of Clans", "Dropbox", "Minecraft"]
ltv_records = []
for app_name in sample_apps:
    ltv_res = engine.forecast_ltv(app_name)
    ltv_records.append({
        "App": ltv_res.app_name,
        "Category": ltv_res.category,
        "Price ($)": ltv_res.price,
        "ARPDAU ($)": ltv_res.arpdau,
        "Est CAC ($)": ltv_res.cac,
        "LTV 30D ($)": ltv_res.ltv_30d,
        "LTV 360D ($)": ltv_res.ltv_360d,
        "LTV/CAC": ltv_res.ltv_cac_ratio,
        "Payback (Days)": ltv_res.payback_period_days,
        "Tier": ltv_res.economic_tier
    })
pd.DataFrame(ltv_records)


## 3. Developer Maintenance Lifecycle & Abandonware Audit

Analyze catalog-wide release recency and identify dormant or abandonware applications requiring intervention.


In [ ]:
df_lifecycle = engine.audit_developer_lifecycle()
tier_counts = df_lifecycle["Lifecycle_Tier"].value_counts().reset_index()
tier_counts.columns = ["Lifecycle_Tier", "App_Count"]
tier_counts["Share_Percent"] = (tier_counts["App_Count"] / len(df_lifecycle) * 100).round(2)
tier_counts


## 4. Holistic App Churn Risk Diagnostic & Prescriptive Playbook

Diagnose composite 0–100 churn scores and review tailored developer mitigation playbooks.


In [ ]:
churn_apps = ["Duolingo: Learn Languages Free", "Subway Surfers", "Colorfy: Colouring Book for Adults"]
for app_name in churn_apps:
    diag = engine.evaluate_churn_risk(app_name)
    print(f"\n=== {app_name} ===")
    print(f"Category: {diag.category} | Rating: {diag.rating} | Days Stale: {diag.days_since_update}")
    print(f"Churn Risk Score: {diag.churn_risk_score}/100 [{diag.churn_risk_tier}]")
    print(f"Primary Driver: {diag.primary_churn_driver}")
    print("Actionable Playbook:")
    for act in diag.prescriptive_playbook:
        print(f"  - {act}")


## 5. Modern Portfolio Theory (Markowitz) Capital Allocation

Find optimal portfolio asset allocation weights across app verticals to maximize risk-adjusted Sharpe ratios.


In [ ]:
port_res = engine.optimize_app_portfolio(["GAME", "FINANCE", "TOOLS", "HEALTH_AND_FITNESS", "EDUCATION"])
print(f"Expected Annual Return: {port_res.expected_annual_return * 100:.2f}%")
print(f"Portfolio Volatility: {port_res.portfolio_volatility * 100:.2f}%")
print(f"Sharpe Ratio: {port_res.sharpe_ratio:.2f}")
print(f"Verdict: {port_res.diversification_verdict}")
print("Optimal Weights:")
for asset, weight in port_res.optimal_weights.items():
    print(f"  * {asset}: {weight * 100:.1f}%")


## 6. Publication Visualizations & Analytical Reports

Render Figure 26 (Cohort Retention & LTV Trajectory) and Figure 27 (Churn Risk & Markowitz Frontier).


In [ ]:
f26, f27 = engine.generate_publication_figures()
reports = engine.generate_all_reports()
print("Generated Figures:", f26.name, f27.name)
print("Generated Reports:", list(reports.keys()))
